In [ ]:
library(Seurat)
library(Matrix)
library(variancePartition)
library(BiocParallel)
library(limma)
library(ggplot2)
library(dplyr)
library(DESeq2)
library(edgeR)

In [ ]:
make_cfg <- function(root_dir,
                     seurat_rds_name    = "seurat_object.rds",
                     raw_counts_name    = "raw_counts.csv",
                     logged_counts_name = "normalized_counts.csv",
                     features_name      = "features_counts.csv",
                     metadata_name      = "cell_metadata.csv",
                     coords_name        = "coords_xy.csv",
                     quint_labels_name  = "Color_key.xlsm") {
  list(
    root_dir      = root_dir,
    seurat_rds    = file.path(root_dir, seurat_rds_name),
    raw_counts    = file.path(root_dir, raw_counts_name),
    logged_counts = file.path(root_dir, logged_counts_name),
    features      = file.path(root_dir, features_name),
    metadata      = file.path(root_dir, metadata_name),
    coords        = file.path(root_dir, coords_name),
    quint_labels  = file.path(root_dir, quint_labels_name)
  )
}

build_seurat_from_folder <- function(cfg, assay_name = "RNA") {
  message(paste0("Loading from: ", cfg$root_dir))
  counts        <- read.csv(cfg$raw_counts,     row.names = 1, check.names = FALSE)
  logged_counts <- read.csv(cfg$logged_counts,  row.names = 1, check.names = FALSE)
  features      <- read.csv(cfg$features,       row.names = 1, check.names = FALSE)
  meta          <- read.csv(cfg$metadata,       row.names = 1, check.names = FALSE)
  coords        <- read.csv(cfg$coords,         row.names = 1, check.names = FALSE)
  if (ncol(counts) != nrow(features))
    stop(paste("Mismatch: Counts matrix has", ncol(counts), "genes, but features file has", nrow(features)))
  colnames(counts)        <- rownames(features)
  colnames(logged_counts) <- rownames(features)
  common_cells <- Reduce(intersect, list(rownames(counts), rownames(meta),
                                         rownames(coords), rownames(logged_counts)))
  if (length(common_cells) == 0)
    stop("No common Cell IDs found. Check your CSV row names.")
  message(paste0("  Matched ", length(common_cells), " cells across all files."))
  counts        <- counts[common_cells, ]
  logged_counts <- logged_counts[common_cells, ]
  meta          <- meta[common_cells, ]
  coords        <- coords[common_cells, ]
  seu <- CreateSeuratObject(counts = t(counts), meta.data = meta, assay = assay_name)
  seu <- SetAssayData(seu, layer = "data", new.data = as.matrix(t(logged_counts)))
  seu <- AddMetaData(seu, metadata = coords)
  return(seu)
}

print("2: Cortex Up")
cfg_file_up <- make_cfg("/path/to/csvs/up")
up <- build_seurat_from_folder(cfg_file_up)

print("3: Cortex Down")
cfg_file_down <- make_cfg("/path/to/csvs/down")
down <- build_seurat_from_folder(cfg_file_down)

In [ ]:
# run_de_dream: region-aware Dream LMM
run_de_dream <- function(seurat_obj, dataset_name, out_prefix, save_dir,
                         region_col = "napari_region") {
  message(paste0("\n>>> [DREAM] STARTING FOR: ", dataset_name))
  num_cts  <- length(unique(seurat_obj$cell_type))
  form_str <- if (num_cts > 1) {
    message(paste0("    Detected ", num_cts, " cell types. Including (1|cell_type)."))
    paste0("~ Treatment + log_depth + (1|", region_col, ") + (1|cell_type) + (1|sample_ID)")
  } else {
    message("    Single cell type — removing (1|cell_type) from formula.")
    paste0("~ Treatment + log_depth + (1|", region_col, ") + (1|sample_ID)")
  }
  form_de <- as.formula(form_str)
  message(paste("    Formula:", deparse(form_de)))
  counts <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
  info   <- seurat_obj@meta.data
  counts <- counts[rowSums(counts) > 0, ]
  dge    <- calcNormFactors(DGEList(counts))
  message("    Calculating voom weights...")
  vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
  fit  <- eBayes(dream(vobj, form_de, info, BPPARAM = param))
  target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
  target_coef <- target_coef[length(target_coef)]
  if (length(target_coef) > 0) {
    de_res       <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
    de_res$Gene  <- rownames(de_res)
    de_res$pct_1 <- rowMeans(counts[rownames(de_res), , drop = FALSE] > 0)
    write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
  }
}

# run_de_blind: region-blind Dream LMM with per-group expression metrics
run_de_blind <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  message(paste0("\n>>> [DREAM BLIND] STARTING FOR: ", dataset_name))
  form_de  <- ~ Treatment + log_depth + (1|sample_ID)
  geneExpr <- as.matrix(GetAssayData(seurat_obj, layer = "data"))
  counts   <- as.matrix(GetAssayData(seurat_obj, layer = "counts"))
  info     <- seurat_obj@meta.data
  tryCatch({
    dge  <- calcNormFactors(DGEList(counts[rowSums(counts) > 0, ]))
    vobj <- voomWithDreamWeights(dge, form_de, info, BPPARAM = param)
    fit  <- eBayes(dream(vobj, form_de, info, BPPARAM = param))
    target_coef <- grep("Treatment", colnames(fit$coefficients), value = TRUE)
    target_coef <- target_coef[length(target_coef)]
    if (length(target_coef) > 0) {
      de_res        <- topTable(fit, coef = target_coef, number = Inf, sort.by = "P", confint = TRUE)
      de_res$Gene   <- rownames(de_res)
      ref_level     <- levels(seurat_obj$Treatment)[1]
      target_level  <- levels(seurat_obj$Treatment)[2]
      cells_ref     <- which(seurat_obj$Treatment == ref_level)
      cells_target  <- which(seurat_obj$Treatment == target_level)
      de_res$pct_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref,    drop = FALSE] > 0)
      de_res$pct_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop = FALSE] > 0)
      de_res$avg_ref    <- rowMeans(geneExpr[rownames(de_res), cells_ref,    drop = FALSE])
      de_res$avg_target <- rowMeans(geneExpr[rownames(de_res), cells_target, drop = FALSE])
      write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
      message(paste("    -> Saved:", file.path(save_dir, paste0(out_prefix, ".csv"))))
    }
  }, error = function(e) message(paste("    !! SKIPPING:", e$message)))
}

# run_pseudobulk_deseq2: pseudobulk DESeq2 validation
run_pseudobulk_deseq2 <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  message(paste0("    >>> [PB] Running Pseudo-bulk DESeq2: ", dataset_name))
  cts <- AggregateExpression(seurat_obj, group.by = "sample_ID", assays = "RNA", slot = "counts")$RNA
  colnames(cts) <- gsub("-", "_", gsub("^g", "", colnames(cts)))
  colData <- seurat_obj@meta.data %>%
    dplyr::select(sample_ID, Treatment) %>%
    dplyr::distinct(sample_ID, .keep_all = TRUE)
  colData <- colData[match(colnames(cts), colData$sample_ID), ]
  rownames(colData) <- colData$sample_ID
  tryCatch({
    dds <- DESeqDataSetFromMatrix(countData = cts, colData = colData, design = ~ Treatment)
    dds <- dds[rowSums(counts(dds)) >= 10, ]
    dds <- DESeq(dds, quiet = TRUE)
    res <- as.data.frame(results(dds))
    res$Gene <- rownames(res)
    write.csv(res, file = file.path(save_dir, paste0("DESEQ2", out_prefix, ".csv")), row.names = FALSE)
    message(paste("    -> Saved PB:", file.path(save_dir, paste0("DESEQ2", out_prefix, ".csv"))))
  }, error = function(e) message(paste("    !! PB FAILED:", e$message)))
}

# run_seurat_lr: Seurat LR test with optional region latent variable
# Set region_col = "napari_region", "quint_region", or NULL (blind).
run_seurat_lr <- function(seurat_obj, dataset_name, out_prefix, save_dir,
                          region_col = NULL) {
  message(paste0("\n>>> [SEURAT LR] STARTING FOR: ", dataset_name))
  Idents(seurat_obj) <- "Treatment"
  ref_level    <- levels(seurat_obj$Treatment)[1]
  target_level <- levels(seurat_obj$Treatment)[2]
  covariates   <- "log_depth"
  if (!is.null(region_col)) {
    if (region_col %in% colnames(seurat_obj@meta.data)) {
      seurat_obj@meta.data[[region_col]] <- as.factor(seurat_obj@meta.data[[region_col]])
      covariates <- c("log_depth", region_col)
    } else {
      message(paste0("    Warning: '", region_col, "' not found in metadata — running blind."))
    }
  }
  message(paste("    Covariates:", paste(covariates, collapse = ", ")))
  tryCatch({
    de_res <- FindMarkers(seurat_obj,
                          ident.1         = target_level,
                          ident.2         = ref_level,
                          test.use        = "LR",
                          latent.vars     = covariates,
                          logfc.threshold = 0,
                          min.pct         = 0,
                          verbose         = FALSE)
    de_res$Gene <- rownames(de_res)
    colnames(de_res)[colnames(de_res) == "pct.1"]      <- "pct_target"
    colnames(de_res)[colnames(de_res) == "pct.2"]      <- "pct_ref"
    colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC"
    colnames(de_res)[colnames(de_res) == "p_val_adj"]  <- "adj.P.Val"
    colnames(de_res)[colnames(de_res) == "p_val"]      <- "P.Value"
    write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
    message(paste("    -> Saved:", file.path(save_dir, paste0(out_prefix, ".csv"))))
  }, error = function(e) message(paste("    !! SEURAT FAILED:", e$message)))
}

# run_wilcox: Wilcoxon rank-sum test via Seurat
run_wilcox <- function(seurat_obj, dataset_name, out_prefix, save_dir) {
  message(paste0("\n>>> [WILCOX] STARTING FOR: ", dataset_name))
  Idents(seurat_obj) <- "Treatment"
  ref_level    <- levels(seurat_obj$Treatment)[1]
  target_level <- levels(seurat_obj$Treatment)[2]
  tryCatch({
    de_res <- FindMarkers(seurat_obj,
                          ident.1         = target_level,
                          ident.2         = ref_level,
                          test.use        = "wilcox",
                          logfc.threshold = 0,
                          min.pct         = 0,
                          verbose         = FALSE)
    de_res$Gene <- rownames(de_res)
    colnames(de_res)[colnames(de_res) == "pct.1"]      <- "pct_target"
    colnames(de_res)[colnames(de_res) == "pct.2"]      <- "pct_ref"
    colnames(de_res)[colnames(de_res) == "avg_log2FC"] <- "logFC"
    colnames(de_res)[colnames(de_res) == "p_val_adj"]  <- "adj.P.Val"
    colnames(de_res)[colnames(de_res) == "p_val"]      <- "P.Value"
    write.csv(de_res, file = file.path(save_dir, paste0(out_prefix, ".csv")), row.names = FALSE)
    message(paste("    -> Saved:", file.path(save_dir, paste0(out_prefix, ".csv"))))
  }, error = function(e) message(paste("    !! WILCOX FAILED:", e$message)))
}

# prepare_metadata: rename FMT -> Treatment, scale depth, drop Cntrl, relevel
prepare_metadata <- function(seurat_obj) {
  names(seurat_obj@meta.data)[names(seurat_obj@meta.data) == "FMT"] <- "Treatment"
  seurat_obj$log_depth <- as.numeric(scale(log10(seurat_obj$nFeature_RNA + 1)))
  seurat_obj <- subset(seurat_obj, subset = Treatment != "Cntrl")
  seurat_obj$Treatment <- as.factor(seurat_obj$Treatment)
  if ("Healthy_FMT" %in% levels(seurat_obj$Treatment))
    seurat_obj$Treatment <- relevel(seurat_obj$Treatment, ref = "Healthy_FMT")
  return(seurat_obj)
}

In [ ]:

param <- SnowParam(workers = 5, type = "SOCK", progressbar = TRUE, exportglobals = FALSE)

base_dir <- "/path/to/lmm_outputs"
dirs <- list(
  global = file.path(base_dir, "Global_CT_Analysis"),
  local  = file.path(base_dir, "Local_Regional_Analysis"),
  pb     = file.path(base_dir, "Pseudobulk_Validation")
)
sapply(dirs, function(x) if (!dir.exists(x)) dir.create(x, recursive = TRUE))

run_analysis_suite <- function(obj, label, file_tag) {
  run_pseudobulk_deseq2(obj,
    dataset_name = paste0(label, "_PB"),
    out_prefix   = paste0(label, "_", file_tag, "_PB"),
    save_dir     = dirs$pb)
  run_de_blind(obj,
    dataset_name = paste0(label, "_dream_blind"),
    out_prefix   = paste0(label, "_", file_tag, "_dream_blind"),
    save_dir     = dirs$local)
  run_de_dream(obj,
    dataset_name = paste0(label, "_dream_napari"),
    out_prefix   = paste0(label, "_", file_tag, "_dream_napari"),
    save_dir     = dirs$global,
    region_col   = "napari_region")
  run_de_dream(obj,
    dataset_name = paste0(label, "_dream_quint"),
    out_prefix   = paste0(label, "_", file_tag, "_dream_quint"),
    save_dir     = dirs$global,
    region_col   = "quint_region")
  run_seurat_lr(obj,
    dataset_name = paste0(label, "_seurat_blind"),
    out_prefix   = paste0(label, "_", file_tag, "_seurat_blind"),
    save_dir     = dirs$global,
    region_col   = NULL)
  run_seurat_lr(obj,
    dataset_name = paste0(label, "_seurat_napari"),
    out_prefix   = paste0(label, "_", file_tag, "_seurat_napari"),
    save_dir     = dirs$global,
    region_col   = "napari_region")
  run_seurat_lr(obj,
    dataset_name = paste0(label, "_seurat_quint"),
    out_prefix   = paste0(label, "_", file_tag, "_seurat_quint"),
    save_dir     = dirs$global,
    region_col   = "quint_region")
  run_wilcox(obj,
    dataset_name = paste0(label, "_wilcoxon"),
    out_prefix   = paste0(label, "_", file_tag, "_wilcoxon"),
    save_dir     = dirs$global)
}

target_cell_types <- c("Astrocytes", "Microglia")

run_ct_loop <- function(data_obj, dataset_label) {
  message(paste0("\n>>> STARTING HIERARCHICAL LOOP: ", dataset_label))
  for (ct in target_cell_types) {
    cell_col <- if (ct == "Astrocytes.cortex.hippocampus") "cell_type" else "ct_simple"
    obj_ct   <- data_obj[, data_obj@meta.data[[cell_col]] == ct]
    if (ncol(obj_ct) < 100) { message(paste("Skipping low cell count:", ct)); next }
    ct_clean <- gsub("[^A-Za-z0-9]", "_", ct)
    message(paste0("   Processing: ", ct))
    run_analysis_suite(obj_ct, label = paste0(dataset_label, "_", ct), file_tag = ct_clean)
    rm(obj_ct); gc()
  }
}

seurat_obj_up   <- prepare_metadata(up)
seurat_obj_down <- prepare_metadata(down)

message("\n>>> RUNNING WHOLE DATASETS...")
run_analysis_suite(seurat_obj_up,   label = "UP",   file_tag = "WHOLE")
run_analysis_suite(seurat_obj_down, label = "DOWN", file_tag = "WHOLE")

message("\n>>> RUNNING CT LOOPS...")
run_ct_loop(seurat_obj_up,   "UP")
run_ct_loop(seurat_obj_down, "DOWN")

message("\n--- ALL ANALYSES COMPLETE ---")